# 01_domain_model — Entity, relations, Lifecycle, Resource, Action together

The full domain picture in one example: entities with fields and lifecycles, bidirectional relations checked at startup (`AssociationOne/Many` with `Inverse`, `CompositeMany` for owned order lines), a `Resource` as a pure transport layer, and a `CreateOrderAction` that uses the resource via `@connection` — logic separate from transport.

This is the shared example referenced by the **Resource**, **Entity**, and **Relations** chapters.

> **Colab note:** the entity cell starts with `from __future__ import annotations` so the forward-referenced relations (a class referring to a sibling defined later) stay lazy until `model_rebuild()`. A `from __future__` import must be the first statement of its cell — keep it at the top of that cell.

In [ ]:
!pip install aoa-action-machine

In [ ]:
from __future__ import annotations

import asyncio
import uuid
from typing import Annotated

from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain import (
    AssociationMany,
    AssociationOne,
    BaseEntity,
    CompositeMany,
    Inverse,
    Lifecycle,
    Rel,
    build,
)
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_float, result_string
from aoa.action_machine.intents.connection import connection
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain and lifecycles (finite-state machines)

In [ ]:
class StoreDomain(BaseDomain):
    name = "store"
    description = "E-commerce store domain"


class OrderLifecycle(Lifecycle):
    _template = (
        Lifecycle()
        .state("new", "New").to("confirmed", "cancelled").initial()
        .state("confirmed", "Confirmed").to("shipped", "cancelled").intermediate()
        .state("shipped", "Shipped").to("delivered").intermediate()
        .state("delivered", "Delivered").final()
        .state("cancelled", "Cancelled").final()
    )


class OrderLineLifecycle(Lifecycle):
    _template = (
        Lifecycle()
        .state("pending", "Pending").to("reserved", "cancelled").initial()
        .state("reserved", "Reserved").to("shipped").intermediate()
        .state("shipped", "Shipped").to("delivered").intermediate()
        .state("delivered", "Delivered").final()
        .state("cancelled", "Cancelled").final()
    )

## Entities with relations and lifecycle

All three are declared in one cell so `Inverse()` can reference partner types; `model_rebuild()` resolves the forward references after all three exist. (This cell starts with `from __future__ import annotations` for that reason.)

In [ ]:
from __future__ import annotations


@entity(description="Registered customer account", domain=StoreDomain)
class CustomerEntity(BaseEntity):
    id: str = Field(description="Customer identifier")
    name: str = Field(description="Display name")
    email: str = Field(description="Contact email")

    # association many — customer exists independently of any order
    orders: Annotated[
        AssociationMany[OrderEntity],
        Inverse(OrderEntity, "customer"),
    ] = Rel(description="Orders placed by this customer")


@entity(description="Customer order", domain=StoreDomain)
class OrderEntity(BaseEntity):
    id: str = Field(description="Order identifier")
    total: float = Field(description="Order total", ge=0)
    currency: str = Field(description="Currency code", default="RUB")
    lifecycle: OrderLifecycle = Field(description="Order lifecycle state")

    # association one-to-one — order and customer exist independently
    customer: Annotated[
        AssociationOne[CustomerEntity],
        Inverse(CustomerEntity, "orders"),
    ] = Rel(description="Customer who placed the order")

    # composition one-to-many — order lines cannot exist without this order
    lines: Annotated[
        CompositeMany[OrderLineEntity],
        Inverse(OrderLineEntity, "order"),
    ] = Rel(description="Line items of the order")


@entity(description="Single line item within an order", domain=StoreDomain)
class OrderLineEntity(BaseEntity):
    id: str = Field(description="Line item identifier")
    product_id: str = Field(description="Product identifier")
    product_name: str = Field(description="Product display name")
    quantity: int = Field(description="Quantity ordered", ge=1)
    unit_price: float = Field(description="Unit price at the time of order", ge=0)
    lifecycle: OrderLineLifecycle = Field(description="Line item lifecycle state")

    # association back to parent order
    order: Annotated[
        AssociationOne[OrderEntity],
        Inverse(OrderEntity, "lines"),
    ] = Rel(description="Parent order")


# Rebuild forward references after all three classes are defined
CustomerEntity.model_rebuild()
OrderEntity.model_rebuild()
OrderLineEntity.model_rebuild()

## Resource — transport layer, no business logic

In [ ]:
@meta(description="In-memory order store (transport only)", domain=StoreDomain)
class OrderResource(BaseResource):
    """In-memory stub; a production version would wrap a SQL/Postgres resource."""

    def __init__(self) -> None:
        self._orders: dict[str, dict] = {}
        self._customers: dict[str, dict] = {
            "cust-001": {"id": "cust-001", "name": "Alice", "email": "alice@example.com"},
        }

    def get_wrapper_class(self) -> type[BaseResource] | None:
        return None  # in-memory stub: direct pass-through, no transaction wrapper

    async def load_customer(self, customer_id: str) -> CustomerEntity:
        row = self._customers.get(customer_id)
        if row is None:
            raise KeyError(f"Customer {customer_id!r} not found")
        return build(row, CustomerEntity)

    async def save_order(self, order_id: str, customer_id: str, total: float, currency: str) -> None:
        self._orders[order_id] = {
            "id": order_id,
            "customer_id": customer_id,
            "total": total,
            "currency": currency,
            "status": "new",
        }

## Params, Result, Action — business logic only; transport via `@connection`

In [ ]:
class CreateOrderParams(BaseParams):
    customer_id: str = Field(description="Customer placing the order")
    lines: list[dict] = Field(description="Line items: [{product_id, name, qty, price}]")
    currency: str = Field(description="Currency code", default="RUB")


class CreateOrderResult(BaseResult):
    order_id: str = Field(description="Created order identifier")
    total: float = Field(description="Order total")
    customer_name: str = Field(description="Customer display name")


@meta(description="Create a new customer order", domain=StoreDomain)
@check_roles(GuestRole)
@connection(OrderResource, key="orders")
class CreateOrderAction(BaseAction[CreateOrderParams, CreateOrderResult]):

    # Each step forwards the fields downstream needs (state is per-step, not cumulative).

    @regular_aspect("Load and validate customer")
    @result_string("customer_name", required=True)
    async def load_customer_aspect(self, params, state, box, connections):
        repo = connections["orders"]
        customer = await repo.load_customer(params.customer_id)
        return {"customer_name": customer.name}

    @regular_aspect("Calculate order total")
    @result_string("customer_name", required=True)
    @result_float("total", required=True, min_value=0)
    async def calculate_total_aspect(self, params, state, box, connections):
        return {
            "customer_name": state["customer_name"],
            "total": sum(item["qty"] * item["price"] for item in params.lines),
        }

    @regular_aspect("Persist order")
    @result_string("customer_name", required=True)
    @result_float("total", required=True, min_value=0)
    @result_string("order_id", required=True)
    async def persist_order_aspect(self, params, state, box, connections):
        repo = connections["orders"]
        order_id = f"ord-{uuid.uuid4().hex[:8]}"
        await repo.save_order(
            order_id=order_id,
            customer_id=params.customer_id,
            total=state["total"],
            currency=params.currency,
        )
        return {
            "customer_name": state["customer_name"],
            "total": state["total"],
            "order_id": order_id,
        }

    @summary_aspect("Build result")
    async def final_summary(self, params, state, box, connections):
        return CreateOrderResult(
            order_id=state["order_id"],
            total=state["total"],
            customer_name=state["customer_name"],
        )

## Run

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
async def main() -> None:
    order_resource = OrderResource()
    machine = ActionProductMachine()

    result = await machine.run(
        Context(),
        CreateOrderAction(),
        CreateOrderParams(
            customer_id="cust-001",
            lines=[
                {"product_id": "sku-1", "name": "Wireless Headphones", "qty": 1, "price": 8990.0},
                {"product_id": "sku-2", "name": "USB-C Cable", "qty": 2, "price": 490.0},
            ],
        ),
        connections={"orders": order_resource},
    )

    print(f"order_id:      {result.order_id}")
    print(f"total:         {result.total} RUB")
    print(f"customer_name: {result.customer_name}")


await main()